# 停止周期重新武装检验
单一固定机制、0拟合、3原生账户。原12%触发阈值和风控仓位约束不变，完成等待/恢复后，回撤重回触发线以内才允许下一个停止周期。此为历史机制诊断，非独立留出验证。

In [1]:
import os,json,qlib
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/stop_rearm_20261014'
qlib.init(provider_uri=str(root/'data/qlib/full_path_tiered_allocation_20261013'),region='us',kernels=1)
from src.workflow.stop_rearm import run
summary=run(root=root,card=card,plan=json.loads((card/'plan.json').read_text()))
summary

backtest loop:   0%|          | 0/3775 [00:00<?, ?it/s]

backtest loop:   0%|          | 0/3775 [00:00<?, ?it/s]

backtest loop:   0%|          | 0/3775 [00:00<?, ?it/s]

{'experiment': 'stop_rearm_20261014',
 'plan_sha256': '0e8d617c794623e937b8aa08c218bca64a566ec479b2b18c7585e637a42b5f5f',
 'decision': 'not_promoted_fixed_rearm_failed',
 'gates': {'net_soxx_excess_all_costs': False,
  'drawdown_30pct': True,
  'cvar_no_worse': True,
  'rolling_70pct': False,
  'annual_70pct': False,
  'bootstrap_positive': False,
  'beat_original_all_costs': False,
  'independent_development': False,
  'qualified_forward': False},
 'objective_achieved': False,
 'new_fits': 0,
 'new_native_cases': 3,
 'reused_native_cases': 6,
 'recorder_id': '1bc047c9ac2141d08b7c51564d6413de',
 'experiment_id': '384376636317382133',
 'recorder_uri': '/Users/rziiiii/dev/workspace/research/stop_rearm_20261014/qlib_recorder',
 'recorder_local_dir': '/Users/rziiiii/dev/workspace/research/stop_rearm_20261014/qlib_recorder/384376636317382133/1bc047c9ac2141d08b7c51564d6413de',
 'provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013',
 'region': 'us',
 'a

# 停止周期重新武装：固定机制未通过

## 机制与来源

0新增拟合，3个新增Qlib原生账户，6个已验证原策略/SOXX对照复用。2023现金起点，10万美元、IBKR Pro Tiered及0/2/5bp独立结算。原数据、模型、现金底线、SOXL上限、12%停止触发线、等待/恢复时长以及紧急/近期损失上限不变。唯一变化：原等待/恢复/持有周期结束后，若已知净值回撤仍不低于12%，继续状态3；回撤低于12%后才回到状态0，允许下一次停止周期。

这是一项研究规则改变，不是修复已确认的软件错误。状态3的q仍为1，其他风控上限照常生效；不制造新模型状态类别。默认代码路径保持原转移行为，生产release未修改。

## 分歧与结果

原策略2bp和5bp首次开盘执行决策分歧为2023-05-30；首次风险状态分歧为2024-02-21，当日回撤分别11.8992%与12.2091%，落在12%触发线两侧。随后共有456个交易日状态不同；不能把全部差异都归因于第一次分歧。

候选与各自原策略在首次状态变化前的逐日收益完全核对：0bp前176日、2/5bp前175日；首次变化分别2023-09-18、2023-09-15、2023-09-15。候选介入早于上面的跨成本首次状态分歧，这是对原完整历史规则的固定改变，而不是只修改事后选中的2024日期。

|滑点bp|方案|累计净收益%|监测最大回撤%|平均期末现金%|停止周期启动次数|
|---:|---|---:|---:|---:|---:|
|0|candidate|247.81|-29.48|47.08|9|
|0|original|228.99|-29.50|48.19|15|
|0|soxx|387.75|-43.38|0.15|0|
|2|candidate|193.44|-28.94|48.77|12|
|2|original|279.67|-27.07|48.31|19|
|2|soxx|387.73|-43.39|0.14|0|
|5|candidate|192.61|-29.00|48.42|11|
|5|original|41.12|-31.42|58.30|31|
|5|soxx|387.70|-43.39|0.12|0|

5bp的停止启动次数从31降至11，收益和回撤均改善；但2bp由19降至12次后，收益降低、回撤增加。减少重复触发不能保证择时更好。0bp改善也没有跨过SOXX基准。

候选2bp在2023/2024/2025/2026逐年相对财富分别约0.8225/0.9674/0.8036/0.9411，全部小于1；5bp同样四年全部落后。开发与独立前向资格未建立，历史区间已复用。直接超额、年度/滚动稳定性、区块置信区间及跨成本优于原策略门均失败；回撤与CVaR门通过。

决定：**not_promoted_fixed_rearm_failed**。稳定净SOXX超额目标未完成。该固定重新武装路线封存，不继续调整阈值、时长或成本参数。

## 风险与下一步

收益是同执行时钟的历史回测，不是实际模拟盘成交；监测回撤仅覆盖原生日收益及已知/收盘点，不等于全分钟风险认证。历史当前费率投影、复权和真实股数/分红现金流资格继承来源限制。默认风控和生产SIM未改。

本轮表明循环规则的稳定化能降低部分成本路径差异，但未产生有效超额。下一项信息需求是原开盘动作模型对“执行还是持有”的净收益优势是否按金额正确排序，尤其概率分类与收益幅度是否错配；先核查已冻结动作标签、成熟时点和训练证据，不再搜索本轮风控参数。

证据：[直接比较](backtest_benchmark_comparison.csv)、[年度](annual_metrics.csv)、[分歧](first_divergence.csv)、[区块区间](uncertainty.csv)、[Recorder清单](verification_manifest.json)。
